# Top-K: BF16 vs FP32 vs Torch

Both kernels receive the same values: BF16 logits for the new variant, promoted to FP32 for the old variant and Torch. Input conversion and allocation are outside timing. Torch runs FP32 activation + top-K + conversion to int32 indices/BF16 weights.

`us` is CUDA-graph replay time per call; `xTorch` is Torch time divided by variant time. `overlap%` measures selected-ID overlap with Torch (ties can differ). `max_abs` and `max_rel%` measure weights against Torch's FP32 scores at the selected IDs. `mass_gap` is the largest per-token loss in selected score sum versus Torch's top-K.

The BF16 tests allow 2% relative selection-score error and 5% weight error; the table reports the actual errors. The legacy FP32 path runs only when `N % 8 == 0`, `E % 256 == 0`, and `K <= E / 32`. BF16 additionally supports 128/384 experts and token tails. Both paths require `K <= 16` and `E <= 1024`.

Run against a checkout containing these changes. A fresh Colab runtime clones the `topk` branch. Restart the notebook kernel if rebuilding an extension already imported in this session.


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

root = next((p for p in (Path.cwd(), Path.cwd().parent, Path.cwd() / "xCaliber")
             if (p / "xcaliber" / "setup.py").is_file()), None)
if root is None:
    root = Path.cwd() / "xCaliber"
    subprocess.run(["git", "clone", "-b", "fmoe", "https://github.com/Pranshu-Bahadur/xCaliber.git", str(root)], check=True)
root = root.resolve()
print(root)


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "ninja", "pytest"], check=True)


In [ ]:
import torch

assert torch.cuda.is_available(), "Select a CUDA GPU runtime"
os.environ["TORCH_CUDA_ARCH_LIST"] = ".".join(map(str, torch.cuda.get_device_capability()))
os.environ.setdefault("MAX_JOBS", "4")
print(torch.cuda.get_device_name(), "sm", os.environ["TORCH_CUDA_ARCH_LIST"], "CUDA", torch.version.cuda)
subprocess.run([sys.executable, str(root / "xcaliber" / "setup.py"), "build_ext", "--inplace"], cwd=root, check=True)


In [ ]:
subprocess.run([sys.executable, "-m", "pytest", str(root / "test" / "test_moe.py"), "-q"], cwd=root, check=True)


In [ ]:
import importlib.util

sys.path.insert(0, str(root))
spec = importlib.util.spec_from_file_location("test_moe", root / "test" / "test_moe.py")
moe_tests = importlib.util.module_from_spec(spec)
spec.loader.exec_module(moe_tests)


In [ ]:
Ns = (8, 16, 16384)
Es = (256, 512)
Ks = (2, 8)
results = moe_tests.benchmark(Ns=Ns, Es=Es, Ks=Ks, repeat=100)
